In [ ]:
# notebooks/02_Pipeline_A_2D-CNN_Log-Mel.ipynb

# --- 1. SETUP E CONFIGURAZIONE ---
import os
import sys
import json
import glob
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Aggiungi src al path per importare i moduli custom
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src import data_loader, models, evaluation

# Configurazione specifica della pipeline
PIPELINE_NAME = "Pipeline_C_Vector"
FEATURE_KEY = None # Non serve una chiave singola
FEATURE_LOADER_FN = data_loader.load_vector_features
MODEL_CREATOR_FN = models.create_1d_cnn_gru_model
MODEL_FILENAME = "model_C_vector.keras"
METADATA_FILENAME = "model_C_vector_metadata.json"

FEATURES_DIR = "../data/processed/features"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

print(f"--- ESECUZIONE: {PIPELINE_NAME} ---")

# --- 2. CARICAMENTO DATI ---
print("\n[Fase 1/5] Caricamento di tutti i dati...")
all_data = {}
fold_dirs = sorted(glob.glob(os.path.join(FEATURES_DIR, "fold*")))
for fold_dir in fold_dirs:
    fold_name = os.path.basename(fold_dir)
    print(f"Caricando {fold_name}...")
    X_fold, y_fold = data_loader.collect_fold_data(
        fold_dir, FEATURE_LOADER_FN, feature_key=FEATURE_KEY
    )
    all_data[fold_name] = (X_fold, y_fold)
print("Caricamento completato.")

# --- 3. CROSS-VALIDATION ---
print("\n[Fase 2/5] Avvio Cross-Validation...")
class_names = data_loader.get_class_map()
num_classes = len(class_names)
input_shape = list(all_data.values())[0][0][0].shape

fold_accuracies = []
all_y_true_cv, all_y_pred_cv = [], []

for i, val_fold_name in enumerate(all_data.keys()):
    print(f"\n--- CV Fold {i+1}/{len(all_data)} (Validation: {val_fold_name}) ---")
    
    # Preparazione dati train/val per questo fold
    X_val, y_val = all_data[val_fold_name]
    train_folds = [data for name, data in all_data.items() if name != val_fold_name]
    X_train = np.vstack([f[0] for f in train_folds])
    y_train = np.concatenate([f[1] for f in train_folds])
    
    # Creazione e training del modello
    model = MODEL_CREATOR_FN(input_shape, num_classes)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=10,  # Aumentato, EarlyStopping gestirà l'arresto
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=0
    )
    
    # Valutazione
    _, acc = model.evaluate(X_val, y_val, verbose=0)
    fold_accuracies.append(acc)
    y_pred = np.argmax(model.predict(X_val), axis=1)
    all_y_true_cv.extend(y_val)
    all_y_pred_cv.extend(y_pred)
    print(f"Accuracy del fold: {acc:.4f}")

mean_acc_cv = np.mean(fold_accuracies)
std_acc_cv = np.std(fold_accuracies)
print(f"\nAccuracy media CV: {mean_acc_cv:.4f} ± {std_acc_cv:.4f}")

# --- 4. ADDESTRAMENTO MODELLO FINALE ---
print("\n[Fase 3/5] Addestramento del modello finale su split 80/20...")
X_train_final, y_train_final, X_test_final, y_test_final = data_loader.get_train_test_split_from_folds(all_data)

# Calcolo pesi per classi sbilanciate
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_final), y=y_train_final)
class_weight_dict = dict(enumerate(class_weights))

final_model = MODEL_CREATOR_FN(input_shape, num_classes)
final_history = final_model.fit(
    X_train_final, y_train_final,
    validation_data=(X_test_final, y_test_final),
    epochs=10,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)
    ],
    verbose=1
)

# --- 5. VALUTAZIONE E SALVATAGGIO ---
print("\n[Fase 4/5] Valutazione del modello finale...")
final_loss, final_accuracy = final_model.evaluate(X_test_final, y_test_final, verbose=0)
print(f"Performance finale sul Test Set:")
print(f"  - Loss: {final_loss:.4f}")
print(f"  - Accuracy: {final_accuracy:.4f}\n")

print("\n[Fase 5/5] Salvataggio del modello e dei metadati...")
# Salva modello
model_path = os.path.join(MODELS_DIR, MODEL_FILENAME)
final_model.save(model_path)
print(f"Modello salvato in: {model_path}")

# Salva metadati
metadata = {
    "pipeline_name": PIPELINE_NAME,
    "feature_key": FEATURE_KEY,
    "model_filename": MODEL_FILENAME,
    "input_shape": input_shape,
    "num_classes": num_classes,
    "class_names": class_names,
    "cv_performance": {"mean_accuracy": mean_acc_cv, "std_accuracy": std_acc_cv},
    "final_test_performance": {"accuracy": final_accuracy, "loss": final_loss}
}
metadata_path = os.path.join(MODELS_DIR, METADATA_FILENAME)
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)
print(f"Metadati salvati in: {metadata_path}")